In [1]:
!pip install geopandas shapely scipy

In [3]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
filename = list(uploaded.keys())[0]
df_nonls = pd.read_csv(filename)
df_nonls.head()


Saving Non_LS.csv to Non_LS.csv


,fid,aspect,curvature,dis_to_fault,dis_to_river,dis_to_thrust,dis_to_road,ndvi,nwdi,roughness,twi,slope,slope_units,X,Y
0,1,73.87893,-0.000413,11629.21,1831.7200,30924.17,16735.21,0.102853,-0.017136,196,5.029370,43.58344,4,552317.3282,3251995
1,2,175.31960,0.000545,14346.96,174.9286,31790.11,18902.40,0.520671,-0.579167,199,7.337051,37.68082,2,547141.1145,3252048
2,3,218.64100,-0.000897,14548.55,308.8689,31905.57,19028.26,0.411644,-0.543978,256,8.393194,50.29310,13,546884.9006,3252101
3,4,53.82940,0.002998,12628.61,1897.3670,31582.42,18112.47,0.263830,-0.367518,188,4.105247,38.52156,34,551037.6606,3252522
4,5,130.36830,-0.000878,11245.88,1083.7440,30804.57,15697.46,0.045926,0.069994,235,7.176542,41.40882,44,553360.3829,3251926


In [5]:
# number of slope units
num_units = df_nonls["slope_units"].nunique()

# samples allocated per slope unit
samples_per_unit = max(1, int(1000 / num_units))
samples_per_unit

df_strat = (
    df_nonls
    .groupby("slope_units")
    .apply(lambda g: g.sample(n=min(samples_per_unit, len(g)),
                              random_state=42))
    .reset_index(drop=True)
)

len(df_strat)



/tmp/ipython-input-611093359.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(samples_per_unit, len(g)),


2299

In [6]:
import numpy as np
from scipy.spatial import cKDTree

coords = df_strat[['X','Y']].values
tree = cKDTree(coords)

min_dist = 80  # meters — adjust as needed
keep = np.ones(len(coords), dtype=bool)

for i, point in enumerate(coords):
    if not keep[i]:
        continue
    idx = tree.query_ball_point(point, min_dist)
    for j in idx:
        if j > i:
            keep[j] = False

df_thinned = df_strat[keep].reset_index(drop=True)
len(df_thinned)


2235

In [7]:
import numpy as np

if len(df_thinned) > 1000:
    df_final = df_thinned.sample(n=1000, random_state=42)
else:
    df_final = df_thinned.copy()

len(df_final)


1000

In [8]:
df_final.to_csv(f"nonls_filtered_1000.csv", index=False)
files.download(f"nonls_filtered_1000.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>